In [ ]:
import streamlit as st
import time
import numpy as np
import pandas as pd
import plotly.express as px
import datetime
from google.cloud import bigquery
import mysql.connector
from dotenv import load_dotenv
from google.oauth2 import service_account
from tableone import TableOne, load_dataset
import re
import seaborn as sns
import matplotlib.pyplot as plt
import time 
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
def identify_columns_with_units(df):
    columns_with_units = []
    for column in df.columns:
        if df[column].dtype == object:  # Considera apenas colunas com strings
            sample_value = df[column].dropna().iloc[0]  # Pega um valor não nulo para amostra
            if isinstance(sample_value, str) and re.search(r'\d', sample_value):
                columns_with_units.append(column)
    return columns_with_units

# Função para remover unidades de medida e deixar apenas o valor numérico
def remove_units(df, columns):
    for column in columns:
        # Usar expressão regular para capturar números com opcional ponto decimal
        df[column] = df[column].str.extract(r'(\d+(\.\d+)?)')[0]
        # Converter para float, ignorando erros
        df[column] = pd.to_numeric(df[column], errors='coerce')
    return df

# verificar se um valor é numérico
def is_number(s):
    if ((s == float) or (s == int)):
        try:
            float(s)
            #print(f'### A VA {s} É NUMERO')
            return True
        except ValueError:
            return False

# verificar se todas as células de uma coluna são numéricas
def convert_columns_to_numeric(df):
    for column in df.columns:
        if all(is_number(val) for val in df[column]):
            #print(f'## CONVERTENDO O TIPO DA COLUNA = {column} para float')
            df[column] = df[column].astype(float)
    return df

# Função para converter colunas para True/False e tipo booleano
def convert_columns_to_boolean(df, columns):
    for col in columns:
        df[col] = df[col].replace({1.0: True, 2.0: False}).astype(bool)
    return df

# Função para converter colunas para True/False e tipo booleano
def convert_columns_to_boolean_zero(df, columns):
    for col in columns:
        df[col] = df[col].replace({0.0: True, 1.0: False}).astype(bool)
    return df

def only_nunbers_tb1(df):
    only_nunbers = [col for col in df.columns if ((df[col].dtype == float) or (df[col].dtype == int))]
    return only_nunbers

def get_col(df):
    all_cols = [col for col in df.columns]
    return all_cols

def only_datas(df):
    datas_only = [col for col in df.columns if pd.api.types.is_datetime64_any_dtype(df[col])]    
    return datas_only

In [ ]:
def conn_sql(query):
    # credentials
    usuario = 
    pwd = 
    host = 
    db = 
    
    load_dotenv() 
    
    key_path = "/home/jimi//dashboard/env.json"
    
    credentials = service_account.Credentials.from_service_account_file(
        filename=key_path, scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )
    
    client = bigquery.Client(credentials=credentials, project=credentials.project_id,)

    cnx = mysql.connector.connect(user=usuario, password=pwd, host=host, database=db)
    
    # Create a cursor object
    cursor = cnx.cursor()
    
    # Execute the query
    cursor.execute(query)
    
    # Fetch all the rows
    results = cursor.fetchall()
    
    # Close the cursor and connection
    cursor.close()
    cnx.close()

    return results

In [ ]:
def conn_bg(sql_statement):

    load_dotenv() 
    
    key_path = ""
    
    credentials = service_account.Credentials.from_service_account_file(
        filename=key_path, scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )
    
    client = bigquery.Client(credentials=credentials, project=credentials.project_id,)

    query = client.query(sql_statement) 

    return query

In [ ]:
try:
    query = f""
    
    results = conn_sql(query)
    
    columns = ["Código do paciente", "Nome paciente", "Data de nascimento", "Sexo", "Criado em"]
    df_paciente = pd.DataFrame(results, columns=columns)
    
    # calculating age
    df_paciente['Idade'] = 0
    information = df_paciente[columns].values.flatten()
    
    new = information.tolist()
    
    c = 0
    for i, r in df_paciente.iterrows():
        dob = pd.to_datetime(new[c+2])
        crt = pd.to_datetime(new[c+4]) #aumenta c tbm
        idade_dias = (crt - dob).days
        df_paciente.at[i, 'Idade'] = idade_dias//360
        c += 5
    
    df_paciente = df_paciente.fillna(0)
    df_paciente['Idade'] = df_paciente['Idade'].astype(int)
    df_paciente = df_paciente.drop(['Data de nascimento'], axis=1)

    df_paciente.head()
    
except Exception as e:
    #st.error(f"Erro: {e}")
    print((f"Erro: {e}"))

In [ ]:
try:
    sql_statement = f""
    
    query = conn_bg(sql_statement)
    
    # sql_statement = f"""
    #     SELECT * FROM `-db_name.db_name.dado_resposta` WHERE cod_unidade_saude=4 
    # """
    # query = client.query(sql_statement)    
    
    rows = []
    
    for result in query.result():
        row = {
            "cod_membro_equipe_saude": result.cod_membro_equipe_saude,
            "cod_paciente": result.cod_paciente,
            "cod_usuario": result.cod_usuario, 
            "cod_visita": result.cod_visita,
            "data_resposta": result.timestamp,
            "variavel": result.variavel,
            "valor_variavel": result.valor_variavel,
        }
        rows.append(row)
     
    columns = ['cod_paciente', 'cod_usuario', 'cod_visita', 'cod_membro_equipe_saude', 'data_resposta', 'variavel', 'valor_variavel']
    df_bq = pd.DataFrame(rows, columns=columns)
    df_bq = df_bq.drop_duplicates(subset=['cod_paciente', 'variavel'], keep='last')
    
    df_pivot = df_bq.pivot_table(index='cod_paciente', columns='variavel', values='valor_variavel', aggfunc='last').reset_index()
    result_df = df_pivot #.drop('cod_paciente', axis=1)
    
    df_paciente = df_paciente.rename(columns={'Código do paciente': 'cod_paciente', 'Sexo': 'sexo', 'Idade': 'idade', 'Criado em':'data_resposta'})
    
    #copia do df_paciente para usar nos filtros de datas
    df_paciente_filto_datas = df_paciente
    
    df_merge_left = pd.merge(result_df, df_paciente, how="left", on=['cod_paciente'])
    
    df_completo = df_merge_left
    df_completo.head()

except Exception as e:
     #st.error(f"Erro: {e}")
     print((f"Erro: {e}"))

In [ ]:
df_completo.head()

In [ ]:
progress_bar = st.progress(0)
status_text = st.empty()

# Consulta para contar o número total de resultados
count_sql_statement = ""

# Executa a consulta e obtém o total de resultados
count_query = conn_bg(count_sql_statement)
total_results = next(count_query.result()).total  # Usando 'next()' para acessar o primeiro resultado

# Consulta principal
sql_statement = f""
query = conn_bg(sql_statement)

rows = []

# Loop para processar os resultados e atualizar a barra de progresso
for i, result in enumerate(query.result()):
    row = {
        "cod_membro_equipe_saude": result.cod_membro_equipe_saude,
        "cod_paciente": result.cod_paciente,
        "cod_usuario": result.cod_usuario, 
        "cod_visita": result.cod_visita,
        "data_resposta": result.timestamp,
        "variavel": result.variavel,
        "valor_variavel": result.valor_variavel,
    }
    rows.append(row)
    
    # Atualizando a barra de progresso
    #progress_bar.progress((i + 1) / total_results)
    print((i + 1) / total_results)

# Limpa a barra de progresso
progress_bar.empty()

columns = []
df_bq = pd.DataFrame(rows, columns=columns)
df_bq = df_bq.drop_duplicates(subset=['cod_paciente', 'variavel'], keep='last')

df_pivot = df_bq.pivot_table(index='cod_paciente', columns='variavel', values='valor_variavel', aggfunc='last').reset_index()
result_df = df_pivot #.drop('cod_paciente', axis=1)

df_paciente = df_paciente.rename(columns={'Código do paciente': 'cod_paciente', 'Sexo': 'sexo', 'Idade': 'idade', 'Criado em':'data_resposta'})

#copia do df_paciente para usar nos filtros de datas
df_paciente_filto_datas = df_paciente

df_merge_left = pd.merge(result_df, df_paciente, how="left", on=['cod_paciente'])

#####################################################################################################
############### df_completo E df_merge_left TEM OS DADOS JUNTOS ENTRE FIREBASE E BIGQ
#####################################################################################################
df_completo = df_merge_left
#####################################################################################################
############### df_completo E df_merge_left TEM OS DADOS JUNTOS ENTRE FIREBASE E BIGQ
#####################################################################################################

#st.write("## DF COMPLETO")
#st.dataframe(df_completo)

#####################################################################################################
############### PEGA OS DADOS DO USUARIO AS VARIAVEIS DO BIGQUERY
#####################################################################################################

df_completo.at[1149, 'sexo'] = 'Feminino'

df_completo = df_completo.dropna(thresh=200, axis=1)

# Identificar colunas com valores numéricos e unidades de medida
columns_to_process = identify_columns_with_units(df_completo)

# Remover unidades de medida e deixar apenas o valor numérico
df_completo = remove_units(df_completo, columns_to_process)

df_completo = convert_columns_to_numeric(df_completo)

columns_to_process = identify_columns_with_units(df_completo)
df_completo = remove_units(df_completo, columns_to_process)
df_completo = convert_columns_to_numeric(df_completo)

# convertendo 0 e 1 para float
columns_to_convert_zero = [
]

# Lista de colunas a serem convertidas
columns_to_convert = [
]

df_completo = convert_columns_to_boolean(df_completo, columns_to_convert)
df_completo = convert_columns_to_boolean_zero(df_completo, columns_to_convert_zero)

#####################################################################################################
############### PREPARANDO PARA A TABELA 1 COM TRESHOLD
#####################################################################################################

df_tabela1 = df_completo
df_tabela1 = df_tabela1.drop(['cod_paciente', 'dxa_preenchido_por', 'Nome paciente'], axis=1)

colunas_com_data = [col for col in df_tabela1.columns if 'data' in col.lower()]

for cols in df_tabela1.columns:
    if cols in colunas_com_data:
        df_tabela1[cols] = pd.to_datetime(df_tabela1[cols], format='mixed')

datas = only_datas(df_tabela1)
columns = get_col(df_tabela1)
continuous = only_nunbers_tb1(df_tabela1)#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
categorical = ['sexo']

#mytable = TableOne(df_tabela1, columns, categorical, continuous)
 
df_tabela_teste_drop_com_trash = df_tabela1.dropna(thresh=600, axis=1)

### tavleone like a DF
table1 = TableOne(df_tabela_teste_drop_com_trash, dip_test=True, normal_test=True, tukey_test=True)
table1

In [ ]:
#####################################################################################################
############### funções aux
#####################################################################################################
# Função para identificar colunas com valores numéricos e unidades de medida
def identify_columns_with_units(df):
    columns_with_units = []
    for column in df.columns:
        if df[column].dtype == object:  # Considera apenas colunas com strings
            sample_value = df[column].dropna().iloc[0]  # Pega um valor não nulo para amostra
            if isinstance(sample_value, str) and re.search(r'\d', sample_value):
                columns_with_units.append(column)
    return columns_with_units

# Função para remover unidades de medida e deixar apenas o valor numérico
def remove_units(df, columns):
    for column in columns:
        # Usar expressão regular para capturar números com opcional ponto decimal
        df[column] = df[column].str.extract(r'(\d+(\.\d+)?)')[0]
        # Converter para float, ignorando erros
        df[column] = pd.to_numeric(df[column], errors='coerce')
    return df

# verificar se um valor é numérico
def is_number(s):
    if ((s == float) or (s == int)):
        try:
            float(s)
            #print(f'### A VA {s} É NUMERO')
            return True
        except ValueError:
            return False

# verificar se todas as células de uma coluna são numéricas
def convert_columns_to_numeric(df):
    for column in df.columns:
        if all(is_number(val) for val in df[column]):
            #print(f'## CONVERTENDO O TIPO DA COLUNA = {column} para float')
            df[column] = df[column].astype(float)
    return df

# Função para converter colunas para True/False e tipo booleano
def convert_columns_to_boolean(df, columns):
    for col in columns:
        df[col] = df[col].replace({1.0: True, 2.0: False}).astype(bool)
    return df

# Função para converter colunas para True/False e tipo booleano
def convert_columns_to_boolean_zero(df, columns):
    for col in columns:
        df[col] = df[col].replace({0.0: True, 1.0: False}).astype(bool)
    return df

def only_nunbers_tb1(df):
    only_nunbers = [col for col in df.columns if ((df[col].dtype == float) or (df[col].dtype == int))]
    return only_nunbers

def get_col(df):
    all_cols = [col for col in df.columns]
    return all_cols

def only_datas(df):
    datas_only = [col for col in df.columns if pd.api.types.is_datetime64_any_dtype(df[col])]    
    return datas_only

In [ ]:
df_paciente = df_paciente.rename(columns={'Código do paciente': 'cod_paciente', 'Sexo': 'sexo', 'Idade': 'idade', 'Criado em':'data_resposta'})

#copia do df_paciente para usar nos filtros de datas
df_paciente_filto_datas = df_paciente

df_merge_left = pd.merge(result_df, df_paciente, how="left", on=['cod_paciente'])

In [ ]:
df_completo = df_merge_left

In [ ]:
df_completo.head()

In [ ]:
for col in df_completo.columns:
    print (col)

In [ ]:
desfechos = [
            ]

# Filtra o DataFrame para as variáveis selecionadas
df_selecionado = df_completo[desfechos]

In [ ]:
print(df_selecionado.to_string())

In [ ]:
df_completo.at[1149, 'sexo'] = 'Feminino'

# Identificar colunas com valores numéricos e unidades de medida
columns_to_process = identify_columns_with_units(df_completo)

# Remover unidades de medida e deixar apenas o valor numérico
df_completo = remove_units(df_completo, columns_to_process)

df_completo = convert_columns_to_numeric(df_completo)

columns_to_process = identify_columns_with_units(df_completo)
df_completo = remove_units(df_completo, columns_to_process)
df_completo = convert_columns_to_numeric(df_completo)

# convertendo 0 e 1 para float
columns_to_convert_zero = [
]

# Lista de colunas a serem convertidas
columns_to_convert = [
]

df_completo = convert_columns_to_boolean(df_completo, columns_to_convert)
df_completo = convert_columns_to_boolean_zero(df_completo, columns_to_convert_zero)

In [ ]:
#####################################################################################################
############### PREPARANDO PARA A TABELA 1 COM TRESHOLD
#####################################################################################################

df_tabela1 = df_completo
df_tabela1 = df_tabela1.drop([], axis=1)

colunas_com_data = [col for col in df_tabela1.columns if 'data' in col.lower()]

for cols in df_tabela1.columns:
    if cols in colunas_com_data:
        df_tabela1[cols] = pd.to_datetime(df_tabela1[cols], format='mixed')

datas = only_datas(df_tabela1)
columns = get_col(df_tabela1)
continuous = only_nunbers_tb1(df_tabela1)#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
categorical = ['sexo']

#mytable = TableOne(df_tabela1, columns, categorical, continuous)
 
df_tabela_teste_drop_com_trash = df_tabela1.dropna(thresh=600, axis=1)

### tavleone like a DF
table1 = TableOne(df_tabela_teste_drop_com_trash, dip_test=True, normal_test=True, tukey_test=True)
table1
#####################################################################################################
############### tratando valores numéricos
#####################################################################################################

In [ ]:
#####################################################################################################
############### tratando valores numéricos
#####################################################################################################
df_completo.at[1149, 'sexo'] = 'Feminino'

# Identificar colunas com valores numéricos e unidades de medida
columns_to_process = identify_columns_with_units(df_completo)

# Remover unidades de medida e deixar apenas o valor numérico
df_completo = remove_units(df_completo, columns_to_process)

df_completo = convert_columns_to_numeric(df_completo)

columns_to_process = identify_columns_with_units(df_completo)
df_completo = remove_units(df_completo, columns_to_process)
df_completo = convert_columns_to_numeric(df_completo)

# convertendo 0 e 1 para float
columns_to_convert_zero = [
]

# Lista de colunas a serem convertidas
columns_to_convert = [
]

df_completo = convert_columns_to_boolean(df_completo, columns_to_convert)
df_completo = convert_columns_to_boolean_zero(df_completo, columns_to_convert_zero)


sql_statement_datas = f"""
"""#.format(data_inicial, data_final)

query = client.query(sql_statement_datas) 

rows = []
for result in query.result():
    row = {
        #'cod_unidade_saude': result.cod_membro_equipe_saude,
        "cod_paciente": result.cod_paciente,
        "cod_usuario": result.cod_usuario, 
        "cod_visita": result.cod_visita,
        "data_resposta": result.timestamp,
        "variavel": result.variavel,
        "valor_variavel": result.valor_variavel,
    }
    rows.append(row)

columns = ['cod_paciente', 'cod_usuario', 'cod_visita', 'data_resposta', 'variavel', 'valor_variavel']
df_bq = pd.DataFrame(rows, columns=columns)
df_bq = df_bq.drop_duplicates(subset=['cod_paciente', 'variavel'], keep='last')

df_pivot = df_bq.pivot_table(index='cod_paciente', columns='variavel', values='valor_variavel', aggfunc='last').reset_index()
result_df = df_pivot #.drop('cod_paciente', axis=1)

df_paciente_filto_datas = df_paciente_filto_datas.rename(columns={'Código do paciente': 'cod_paciente', 'Sexo': 'sexo', 'Idade': 'idade'})

df_merge_left = pd.merge(result_df, df_paciente_filto_datas, how="left", on=['cod_paciente'])

df_filtro_datas = df_merge_left

# Primeiro, agrupe os dados por dia e conte o número de consultas por dia
consultas_por_dia = df_filtro_datas.groupby(df_filtro_datas["data_resposta"].dt.date).size().reset_index(name='quantidade_consultas')

In [ ]:
consultas_por_dia

In [ ]:
#####################################################################################################
############### por questionário
#####################################################################################################

#patient data
query = f""

cnx = mysql.connector.connect(user=usuario, password=pwd, host=host, database=db)

# Create a cursor object
cursor = cnx.cursor()

# Execute the query
cursor.execute(query)

# Fetch all the rows
results = cursor.fetchall()

# Close the cursor and connection
cursor.close()
cnx.close()
  
columns = [""]

df_paciente = pd.DataFrame(results, columns=columns)

df_quests = df_paciente

# Agrupa os dados pelo nome do questionário e conta o número de ocorrências
questionario_counts = df_quests['name'].value_counts().reset_index()
questionario_counts.columns = ['name', 'count']
 
# Primeiro, agrupe os dados por dia e conte o número de consultas por dia
quests_por_dia = df_quests.groupby(df_quests["dt_inicio_visita"].dt.date).size().reset_index(name='quantidade_quests')

# Remover a hora e manter apenas ano, mês e dia
quests_por_dia["dt_inicio_visita"] = pd.to_datetime(quests_por_dia["dt_inicio_visita"]).dt.floor('D')

# Agrupando por mês e ano e contando a quantidade de consultas
quests_por_mes = df_quests.groupby(df_quests["dt_inicio_visita"].dt.to_period('M')).size().reset_index(name='quantidade_quests')
quests_por_mes["dt_inicio_visita"] = quests_por_mes["dt_inicio_visita"].dt.to_timestamp()

date = '2015-12-30'
date64 = np.datetime64(date)

#exclui as linhas com data < 2015-12-30
quests_por_mes = quests_por_mes.drop(quests_por_mes[quests_por_mes['dt_inicio_visita'] <= date64].index)

In [ ]:
print(quests_por_mes['dt_inicio_visita'].to_string())

In [ ]:
df_completo.info()

In [ ]:
df_completo

In [ ]:


### frontend

#pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

st.set_page_config(
    page_title="",
    page_icon="📋",
    layout="wide",
    initial_sidebar_state="expanded")

st.title('')

# Sidebar for selecting variables
st.sidebar.title('Menu principal')

with st.spinner("Carregando as variáveis"):
 variables = df_completo.columns.tolist() #lista que vem do dataset
 selected_variable = st.sidebar.selectbox('Escolha a variável para ver as estatisticas sobre ela', variables)

col1, col2 = st.columns(2)
col3, col4 = st.columns(2)

# Main page
st.title('Statistical Measures Dashboard')

with st.spinner("Carregando a Tabela 1"):
 df_counts = df_completo[columns_to_convert].apply(pd.Series.value_counts).T
 col1.write('TableOne - Trashold de {}%')
 col1.write(table1.tableone, unsafe_allow_html=True, use_container_width=True)

with st.spinner("Carregando os dados da variável escolhida"):
 if selected_variable:
     # Calculate statistics for the selected variable
     mean = variables[selected_variable].mean()
     median = variables[selected_variable].median()
     std_dev = variables[selected_variable].std()
     variance = variables[selected_variable].var()
     data_range = variables[selected_variable].max() - data[selected_variable].min()
     
     # Display statistics
     
     st.write(f"**Selected Variable:** {selected_variable}")
     st.write(f"**Mean:** {mean:.2f}")
     st.write(f"**Median:** {median:.2f}")
     st.write(f"**Standard Deviation:** {std_dev:.2f}")
     st.write(f"**Variance:** {variance:.2f}")
     st.write(f"**Range:** {data_range:.2f}")

In [ ]:
variaveis_numericas = df_completo.select_dtypes(include=['float', 'int']).columns.tolist()
variaveis_categóricas = df_completo.select_dtypes(include=['bool']).columns.tolist()

In [ ]:
variaveis_numericas

In [ ]:
variaveis_categóricas

In [ ]:
variaveis_numericas = df_completo.select_dtypes(include=['float', 'int']).columns.tolist()
variaveis_categóricas = df_completo.select_dtypes(include=['bool']).columns.tolist()

In [ ]:
var_numerica = df_completo.select_dtypes(include=['float', 'int']).columns.tolist() #lista que vem do dataset
#selected_variable = st.sidebar.selectbox('Escolha a variável para ver as estatisticas sobre ela', variables)
#st.markdown("<h1 style='text-align: center; color: grey;'>Dashboard Aterolab</h1>", unsafe_allow_html=True)
selected_variables_numeric = st.sidebar.multiselect('Escolha as variáveis para exibir ESTATÍSTICAS', var_numerica)

var_categrica = df_completo.select_dtypes(include=['bool']).columns.tolist() #lista que vem do dataset
#selected_variable = st.sidebar.selectbox('Escolha a variável para ver as estatisticas sobre ela', var_cat)
selected_var_cat = st.sidebar.multiselect('Escolha as variáveis para exibir ESTATÍSTICAS', var_categrica)

In [ ]:
selected_variables_numeric

In [ ]:
print(df_completo.shape)

In [ ]:
with st.spinner("Carregando os dados dos QUESTIONÁRIOS"):
  #patient data
  query = 
  
  cnx = mysql.connector.connect(user=usuario, password=pwd, host=host, database=db)
  
  # Create a cursor object
  cursor = cnx.cursor()
  
  # Execute the query
  cursor.execute(query)
  
  # Fetch all the rows
  results = cursor.fetchall()
  
  # Close the cursor and connection
  cursor.close()
  cnx.close()
  
  columns = ['']
  
  df_paciente = pd.DataFrame(results, columns=columns)
  
  df_quests = df_paciente
  
  # Agrupa os dados pelo nome do questionário e conta o número de ocorrências
  questionario_counts = df_quests['name'].value_counts().reset_index()
  questionario_counts.columns = ['name', 'count']
  
  # Primeiro, agrupe os dados por dia e conte o número de consultas por dia
  quests_por_dia = df_quests.groupby(df_quests["dt_inicio_visita"].dt.date).size().reset_index(name='quantidade_quests')
  
  # Remover a hora e manter apenas ano, mês e dia
  quests_por_dia["dt_inicio_visita"] = pd.to_datetime(quests_por_dia["dt_inicio_visita"]).dt.floor('D')
  
  # Agrupando por mês e ano e contando a quantidade de consultas
  quests_por_mes = df_quests.groupby(df_quests["dt_inicio_visita"].dt.to_period('M')).size().reset_index(name='quantidade_quests')
  quests_por_mes["dt_inicio_visita"] = quests_por_mes["dt_inicio_visita"].dt.to_timestamp()
  
  date = '2015-12-30'
  date64 = np.datetime64(date)
  
  #exclui as linhas com data < 2015-12-30
  quests_por_mes = quests_por_mes.drop(quests_por_mes[quests_por_mes['dt_inicio_visita'] <= date64].index)

  quests_por_mes

In [ ]:
quests_por_mes

In [ ]:
var_numerica = df_completo.select_dtypes(include=['float', 'int']).columns.tolist() #lista que vem do dataset
#selected_variable = st.sidebar.selectbox('Escolha a variável para ver as estatisticas sobre ela', variables)
#st.markdown("<h1 style='text-align: center; color: grey;'>Dashboard Aterolab</h1>", unsafe_allow_html=True)
selected_variables_numeric = st.sidebar.multiselect('Escolha as variáveis para exibir ESTATÍSTICAS', var_numerica)

var_categrica = df_completo.select_dtypes(include=['bool']).columns.tolist() #lista que vem do dataset
#selected_variable = st.sidebar.selectbox('Escolha a variável para ver as estatisticas sobre ela', var_cat)
selected_var_cat = st.sidebar.multiselect('Escolha as variáveis para exibir ESTATÍSTICAS', var_categrica)

In [ ]:
stats_list = []

for variable in selected_variables_numeric:
    # Calcula as estatísticas para a variável selecionada
    mean = df_completo[variable].mean()
    median = df_completo[variable].median()
    std_dev = df_completo[variable].std()
    variance = df_completo[variable].var()
    data_range = df_completo[variable].max() - df_completo[variable].min()

    # Adiciona as estatísticas à lista como um dicionário
    stats_list.append({
        "Variável": variable,
        "Média": f"{mean:.2f}",
        "Mediana": f"{median:.2f}",
        "Desvio Padrão": f"{std_dev:.2f}",
        "Variância": f"{variance:.2f}",
        "Intervalo": f"{data_range:.2f}"
    })

# Cria um DataFrame a partir da lista de dicionários
stats_df = pd.DataFrame(stats_list)

# Exibe o DataFrame no Streamlit
col2.write("### ESTATÍSTICAS das Variáveis Numéricas")
col2.dataframe(stats_df)

In [ ]:
stats_df

In [ ]:
df_completo.info()

In [ ]:
for col in df_completo.columns:
    print(col)

In [ ]:
result_df.info()

In [ ]:
identify_columns_with_units(result_df)

columns_to_process = identify_columns_with_units(result_df)

result_df = convert_columns_to_numeric(result_df)

In [ ]:
result_df_corr

In [ ]:
# Filtra o DataFrame com base no intervalo de datas selecionado
start_date = st.sidebar.date_input('Data de início', df_completo['data_resposta'].min())
end_date = st.sidebar.date_input('Data de fim', df_completo['data_resposta'].max())

# Filtra o DataFrame com base no intervalo de datas selecionado
df_filtrado = df_completo[(df_completo['data_resposta'] >= pd.to_datetime(start_date)) & 
                          (df_completo['data_resposta'] <= pd.to_datetime(end_date))]

# Filtrando apenas as colunas numéricas
df_numerico = df_filtrado.select_dtypes(include=['float64', 'int64'])

# Resetando o índice da matriz de correlação para garantir que os rótulos apareçam
correlation_matrix = df_numerico.corr()
correlation_matrix_reset = correlation_matrix.reset_index()

correlation_matrix_reset

In [ ]:
m_o_direita_3 = df_completo['m_o_direita_3']
pa_sistolica = df_completo['pa_sistolica'] 
weight = df_completo['weight']

In [ ]:
pa_sistolica

In [ ]:
stats_df

In [ ]:
# Supondo que df_completo seja seu DataFrame e tenha uma coluna de data chamada 'data_coluna'
df_completo = pd.DataFrame({
    'data_coluna': pd.date_range(start='2023-01-01', periods=365, freq='D'),
    'variavel_1': range(365),
    'variavel_2': range(365, 730)
})

# Sidebar para seleção do intervalo de datas
st.sidebar.header("Filtro por Data")
start_date = st.sidebar.date_input("Data de Início", value=pd.to_datetime("2023-01-01"))
end_date = st.sidebar.date_input("Data de Fim", value=pd.to_datetime("2023-12-31"))